# Vesuvius Challenge 2025 - Topology-Optimized Submission

## Goal: Break 0.550 LB through post-processing

### Key Insight
- Val Dice 0.60 → LB 0.550
- Val Dice 0.63 → LB 0.543 (WORSE!)
- **Dice optimization hurts topology**

### LB Score Composition
```
LB = 0.30 × TopoScore + 0.35 × SurfaceDice + 0.35 × VOI_score
```
- **65% is topology-related** (TopoScore + VOI)

### Post-Processing Strategies
1. **Hysteresis thresholding** - preserves thin connections
2. **Skeleton preservation** - maintains connectivity
3. **VOI-aware filtering** - fixes split/merge errors
4. **Lower threshold (0.3)** - your finding that works

In [ ]:
!pip install tifffile imagecodecs connected-components-3d scikit-image -q

In [ ]:
import os
import gc
import json
import time
import zipfile
import warnings
from pathlib import Path
from typing import Tuple, List

import numpy as np
import pandas as pd
import tifffile
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

from scipy import ndimage
from scipy.ndimage import distance_transform_edt, binary_dilation, binary_erosion
from skimage.morphology import skeletonize_3d, remove_small_objects

try:
    import cc3d
    USE_CC3D = True
except ImportError:
    USE_CC3D = False
    print('cc3d not found, using scipy')

warnings.filterwarnings('ignore')

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
# =============================================================================
# CONFIGURATION (TOP-3 WEIGHTED FOLD ENSEMBLE)
# =============================================================================

class Config:
    TEST_IMAGES_DIR = Path('/kaggle/input/vesuvius-challenge-surface-detection/test_images')
    TEST_CSV = Path('/kaggle/input/vesuvius-challenge-surface-detection/test.csv')

    # Fold artifacts exported by training notebook
    CHECKPOINT_ROOT = Path('/kaggle/input/vesuvius-v9-cv/checkpoints_v9_cv')
    FOLD_SCORES_CSV = Path('/kaggle/input/vesuvius-v9-cv/metrics_v9_cv/fold_scores.csv')
    BEST_POSTPROCESS_JSON = Path('/kaggle/input/vesuvius-v9-cv/metrics_v9_cv/best_postprocess.json')

    OUTPUT_DIR = Path('/kaggle/working/submission_masks')
    SUBMISSION_ZIP = Path('/kaggle/working/submission.zip')

    # Architecture (must match training)
    PATCH_SIZE: Tuple[int, int, int] = (128, 128, 128)
    FEATURES: List[int] = [32, 64, 128, 256, 320, 320]
    N_BLOCKS: List[int] = [1, 2, 3, 5, 5, 5]
    USE_SCSE: bool = True
    USE_DEEP_SUPERVISION: bool = False

    # Inference defaults
    OVERLAP: float = 0.5
    TOP_K_FOLDS: int = 3
    TTA_MODE: str = 'none'   # 'none' or 'zflip'

    DEVICE: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    USE_AMP: bool = True

    # Runtime gate
    RUNTIME_GATE_CASES: int = 5
    RUNTIME_HOURS_LIMIT: float = 8.5
    MIN_OVERLAP: float = 0.4

    # Postprocess defaults (can be overridden by best_postprocess.json)
    PP_THRESHOLD: float = 0.30
    PP_USE_HYSTERESIS: bool = True
    PP_HYST_LOW: float = 0.16
    PP_HYST_HIGH: float = 0.38
    PP_MIN_COMPONENT_SIZE: int = 50
    PP_CLOSING_ITERS: int = 0


cfg = Config()
cfg.OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

if cfg.BEST_POSTPROCESS_JSON.exists():
    payload = json.loads(cfg.BEST_POSTPROCESS_JSON.read_text())
    best = payload.get('best_params', {})
    cfg.PP_THRESHOLD = float(best.get('threshold', cfg.PP_THRESHOLD))
    cfg.PP_USE_HYSTERESIS = bool(best.get('use_hysteresis', cfg.PP_USE_HYSTERESIS))
    cfg.PP_HYST_LOW = float(best.get('hysteresis_low', cfg.PP_HYST_LOW))
    cfg.PP_HYST_HIGH = float(best.get('hysteresis_high', cfg.PP_HYST_HIGH))
    cfg.PP_MIN_COMPONENT_SIZE = int(best.get('min_component_size', cfg.PP_MIN_COMPONENT_SIZE))
    cfg.PP_CLOSING_ITERS = int(best.get('closing_iters', cfg.PP_CLOSING_ITERS))

print('=' * 80)
print('TOPOLOGY SUBMISSION CONFIG')
print('=' * 80)
print(f'Patch={cfg.PATCH_SIZE}, overlap={cfg.OVERLAP}, tta={cfg.TTA_MODE}')
print(f'Top-k folds={cfg.TOP_K_FOLDS}, runtime gate={cfg.RUNTIME_HOURS_LIMIT}h')
print('Postprocess:', {
    'threshold': cfg.PP_THRESHOLD,
    'use_hysteresis': cfg.PP_USE_HYSTERESIS,
    'hysteresis_low': cfg.PP_HYST_LOW,
    'hysteresis_high': cfg.PP_HYST_HIGH,
    'min_component_size': cfg.PP_MIN_COMPONENT_SIZE,
    'closing_iters': cfg.PP_CLOSING_ITERS,
})
print('=' * 80)


In [ ]:
# =============================================================================
# POST-PROCESSING
# =============================================================================

def connected_components_3d(volume, connectivity=26):
    if USE_CC3D:
        return cc3d.connected_components(volume.astype(np.uint8), connectivity=connectivity)
    struct = ndimage.generate_binary_structure(3, 3 if connectivity == 26 else 1)
    labeled, _ = ndimage.label(volume.astype(bool), structure=struct)
    return labeled


def hysteresis_threshold_3d(prob_map, low=0.16, high=0.38):
    high_mask = prob_map >= high
    low_mask = prob_map >= low
    labeled = connected_components_3d(low_mask, connectivity=26)
    keep_labels = np.unique(labeled[high_mask])
    keep_labels = keep_labels[keep_labels > 0]
    return np.isin(labeled, keep_labels).astype(np.uint8)


def apply_postprocess(prob_map, threshold=0.30, use_hysteresis=True, hysteresis_low=0.16,
                      hysteresis_high=0.38, min_component_size=50, closing_iters=0):
    if use_hysteresis:
        binary = hysteresis_threshold_3d(prob_map, low=hysteresis_low, high=hysteresis_high)
    else:
        binary = (prob_map >= threshold).astype(np.uint8)

    if min_component_size > 0:
        labeled = connected_components_3d(binary, connectivity=26)
        if labeled.max() > 0:
            out = np.zeros_like(binary, dtype=np.uint8)
            for i in range(1, int(labeled.max()) + 1):
                c = labeled == i
                if c.sum() >= min_component_size:
                    out[c] = 1
            binary = out

    if closing_iters > 0:
        struct = ndimage.generate_binary_structure(3, 1)
        binary = ndimage.binary_closing(binary, structure=struct, iterations=int(closing_iters)).astype(np.uint8)

    return binary.astype(np.uint8)


print('Post-processing ready')


In [ ]:
# =============================================================================
# MODEL ARCHITECTURE (SafeInstanceNorm3d)
# =============================================================================

class SafeInstanceNorm3d(nn.Module):
    def __init__(self, num_features, eps=1e-5, affine=True):
        super().__init__()
        self.num_features = num_features
        self.eps = eps
        self.affine = affine
        if affine:
            self.weight = nn.Parameter(torch.ones(num_features))
            self.bias = nn.Parameter(torch.zeros(num_features))

    def forward(self, x):
        mean = x.mean(dim=[2, 3, 4], keepdim=True)
        var = x.var(dim=[2, 3, 4], keepdim=True, unbiased=False)
        x_norm = (x - mean) / torch.sqrt(torch.clamp(var, min=self.eps) + self.eps)
        if self.affine:
            x_norm = self.weight.view(1, -1, 1, 1, 1) * x_norm + self.bias.view(1, -1, 1, 1, 1)
        return x_norm


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=True),
            SafeInstanceNorm3d(out_ch),
            nn.LeakyReLU(0.01, inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


class ResBlock(nn.Module):
    def __init__(self, channels, n_convs=2):
        super().__init__()
        self.blocks = nn.Sequential(*[ConvBlock(channels, channels) for _ in range(n_convs)])

    def forward(self, x):
        return x + self.blocks(x)


class scSEBlock(nn.Module):
    def __init__(self, channels, reduction=2):
        super().__init__()
        self.cse_pool = nn.AdaptiveAvgPool3d(1)
        self.cse_fc1 = nn.Linear(channels, channels // reduction)
        self.cse_fc2 = nn.Linear(channels // reduction, channels)
        self.sse_conv = nn.Conv3d(channels, 1, 1)

    def forward(self, x):
        b, c, d, h, w = x.shape
        cse = self.cse_pool(x).view(b, c)
        cse = F.relu(self.cse_fc1(cse))
        cse = torch.sigmoid(self.cse_fc2(cse)).view(b, c, 1, 1, 1)
        sse = torch.sigmoid(self.sse_conv(x))
        return x * cse + x * sse


class ResEncUNet3D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, features=None, n_blocks=None, 
                 use_scse=True, use_deep_supervision=True):
        super().__init__()
        if features is None:
            features = [32, 64, 128, 256, 320, 320]
        if n_blocks is None:
            n_blocks = [1, 3, 4, 6, 6, 6]
        
        self.features = features
        self.use_scse = use_scse
        self.use_deep_supervision = use_deep_supervision
        
        # Encoder
        self.encoders = nn.ModuleList()
        self.attentions = nn.ModuleList()
        self.pools = nn.ModuleList()
        
        for i, (feat, nb) in enumerate(zip(features, n_blocks)):
            in_channels = in_ch if i == 0 else features[i - 1]
            self.encoders.append(nn.Sequential(
                ConvBlock(in_channels, feat),
                *[ResBlock(feat) for _ in range(nb)]
            ))
            self.attentions.append(scSEBlock(feat) if use_scse else nn.Identity())
            if i < len(features) - 1:
                self.pools.append(nn.Conv3d(feat, feat, 2, stride=2))
        
        # Decoder
        self.ups = nn.ModuleList()
        self.dec_convs = nn.ModuleList()
        
        for i in range(len(features) - 2, -1, -1):
            self.ups.append(nn.ConvTranspose3d(features[i+1], features[i], 2, stride=2))
            self.dec_convs.append(ConvBlock(features[i] * 2, features[i]))
        
        self.final = nn.Conv3d(features[0], out_ch, 1)
        
        if use_deep_supervision:
            self.ds_heads = nn.ModuleList([
                nn.Conv3d(features[-(i+2)], out_ch, 1) for i in range(min(4, len(features)-1))
            ])

    def forward(self, x):
        enc_features = []
        for i, (enc, att) in enumerate(zip(self.encoders, self.attentions)):
            x = enc(x)
            x = att(x)
            enc_features.append(x)
            if i < len(self.pools):
                x = self.pools[i](x)
        
        enc_features = enc_features[::-1]
        x = enc_features[0]
        ds_outputs = []
        
        for i, (up, dec) in enumerate(zip(self.ups, self.dec_convs)):
            x = up(x)
            skip = enc_features[i + 1]
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            x = torch.cat([x, skip], dim=1)
            x = dec(x)
            if self.use_deep_supervision and self.training and i < len(self.ds_heads):
                ds_outputs.append(self.ds_heads[i](x))
        
        out = self.final(x)
        if self.use_deep_supervision and self.training:
            return {'output': out, 'deep': ds_outputs}
        return out


print("Model defined")

In [ ]:
# =============================================================================
# INFERENCE (SINGLE MODEL + WEIGHTED ENSEMBLE)
# =============================================================================

def normalize_volume(volume):
    vol = volume.astype(np.float32)
    return (vol - vol.mean()) / (vol.std() + 1e-8)


@torch.no_grad()
def sliding_window_inference_single(model, volume, patch_size=(128, 128, 128),
                                    overlap=0.5, device='cuda', use_amp=True):
    model.eval()
    D, H, W = volume.shape
    pd, ph, pw = patch_size
    sd = max(1, int(pd * (1 - overlap)))
    sh = max(1, int(ph * (1 - overlap)))
    sw = max(1, int(pw * (1 - overlap)))

    pred_sum = np.zeros((D, H, W), dtype=np.float32)
    count = np.zeros((D, H, W), dtype=np.float32)

    sigma = 0.125
    gz = np.exp(-0.5 * ((np.arange(pd) - pd / 2) / (pd * sigma)) ** 2)
    gy = np.exp(-0.5 * ((np.arange(ph) - ph / 2) / (ph * sigma)) ** 2)
    gx = np.exp(-0.5 * ((np.arange(pw) - pw / 2) / (pw * sigma)) ** 2)
    gauss = (gz[:, None, None] * gy[None, :, None] * gx[None, None, :]).astype(np.float32)

    z_pos = list(range(0, max(1, D - pd + 1), sd))
    if D > pd and (D - pd) not in z_pos:
        z_pos.append(D - pd)
    y_pos = list(range(0, max(1, H - ph + 1), sh))
    if H > ph and (H - ph) not in y_pos:
        y_pos.append(H - ph)
    x_pos = list(range(0, max(1, W - pw + 1), sw))
    if W > pw and (W - pw) not in x_pos:
        x_pos.append(W - pw)

    vol_norm = normalize_volume(volume)

    for z in tqdm(z_pos, desc='Inference', leave=False):
        for y in y_pos:
            for x in x_pos:
                patch = vol_norm[z:z + pd, y:y + ph, x:x + pw].copy()
                actual = patch.shape

                if patch.shape != (pd, ph, pw):
                    patch = np.pad(
                        patch,
                        [(0, pd - patch.shape[0]), (0, ph - patch.shape[1]), (0, pw - patch.shape[2])],
                        mode='reflect',
                    )

                inp = torch.from_numpy(patch[None, None].astype(np.float32)).to(device)
                with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
                    out = model(inp)
                    if isinstance(out, dict):
                        out = out['output']
                    pred = torch.sigmoid(out).squeeze().cpu().numpy()

                out = pred[:actual[0], :actual[1], :actual[2]]
                w = gauss[:actual[0], :actual[1], :actual[2]]

                pred_sum[z:z + actual[0], y:y + actual[1], x:x + actual[2]] += out * w
                count[z:z + actual[0], y:y + actual[1], x:x + actual[2]] += w

    return pred_sum / np.maximum(count, 1e-8)


@torch.no_grad()
def model_inference_with_tta(model, volume, patch_size, overlap, device, use_amp, tta_mode='none'):
    base = sliding_window_inference_single(model, volume, patch_size, overlap, device, use_amp)
    if tta_mode == 'zflip':
        flip = sliding_window_inference_single(model, volume[::-1].copy(), patch_size, overlap, device, use_amp)
        base = 0.5 * (base + flip[::-1].copy())
    return base


@torch.no_grad()
def weighted_ensemble_inference(models, weights, volume, cfg, overlap=None, tta_mode=None):
    overlap = cfg.OVERLAP if overlap is None else overlap
    tta_mode = cfg.TTA_MODE if tta_mode is None else tta_mode

    weights = np.array(weights, dtype=np.float64)
    weights = weights / weights.sum()

    ensemble = np.zeros(volume.shape, dtype=np.float32)
    for model, w in zip(models, weights):
        prob = model_inference_with_tta(
            model,
            volume,
            patch_size=cfg.PATCH_SIZE,
            overlap=overlap,
            device=cfg.DEVICE,
            use_amp=cfg.USE_AMP,
            tta_mode=tta_mode,
        )
        ensemble += (w * prob).astype(np.float32)
    return ensemble


print('Inference functions ready')


In [ ]:
# =============================================================================
# LOAD TOP-3 FOLD MODELS + NORMALIZED WEIGHTS
# =============================================================================

def clean_state_dict(state_dict):
    return {k.replace('module.', '').replace('_orig_mod.', ''): v for k, v in state_dict.items()}


def _read_fold_scores(cfg):
    if cfg.FOLD_SCORES_CSV.exists():
        return pd.read_csv(cfg.FOLD_SCORES_CSV)

    rows = []
    for fd in sorted(cfg.CHECKPOINT_ROOT.glob('fold_*')):
        fold = int(fd.name.split('_')[-1])
        ckpt = fd / 'best_model.pth'
        if ckpt.exists():
            c = torch.load(ckpt, map_location='cpu', weights_only=False)
            m = c.get('metrics', {})
            rows.append({
                'fold': fold,
                'best_lb_score': m.get('lb_score', np.nan),
                'best_selection_score': m.get('selection_score', np.nan),
            })
    return pd.DataFrame(rows)


def load_top_fold_models(cfg):
    score_df = _read_fold_scores(cfg)
    if score_df.empty:
        raise RuntimeError('No fold scores/checkpoints found')

    rank_col = 'best_lb_score' if 'best_lb_score' in score_df.columns and score_df['best_lb_score'].notna().any() else 'best_selection_score'
    score_df = score_df.sort_values(rank_col, ascending=False).reset_index(drop=True)
    top_df = score_df.head(cfg.TOP_K_FOLDS).copy()

    raw = top_df[rank_col].fillna(0).values.astype(np.float64)
    if np.all(raw <= 0):
        w = np.ones(len(top_df), dtype=np.float64) / len(top_df)
    else:
        raw = np.clip(raw, a_min=0, a_max=None)
        w = raw / raw.sum()

    models, folds, weights = [], [], []
    for (_, row), wi in zip(top_df.iterrows(), w):
        fold = int(row['fold'])
        ckpt_path = cfg.CHECKPOINT_ROOT / f'fold_{fold}' / 'best_model.pth'
        if not ckpt_path.exists():
            print(f'Warning: missing checkpoint for fold {fold}')
            continue

        model = ResEncUNet3D(
            features=cfg.FEATURES,
            n_blocks=cfg.N_BLOCKS,
            use_scse=cfg.USE_SCSE,
            use_deep_supervision=cfg.USE_DEEP_SUPERVISION,
        ).to(cfg.DEVICE)

        ckpt = torch.load(ckpt_path, map_location=cfg.DEVICE, weights_only=False)
        state = clean_state_dict(ckpt.get('model_state_dict', ckpt))
        model.load_state_dict(state, strict=False)
        model.eval()

        models.append(model)
        folds.append(fold)
        weights.append(float(wi))

    if not models:
        raise RuntimeError('No models loaded')

    weights = np.array(weights, dtype=np.float64)
    weights = (weights / weights.sum()).tolist()

    print('Loaded folds:', folds)
    print('Weights:', [round(x, 6) for x in weights])
    return models, weights, top_df


test_df = pd.read_csv(cfg.TEST_CSV)
print(f'Test volumes: {len(test_df)}')
models, fold_weights, fold_rank_df = load_top_fold_models(cfg)


In [ ]:
# =============================================================================
# RUN SUBMISSION
# =============================================================================

def _infer_mask_dtype():
    train_label_dir = Path('/kaggle/input/vesuvius-challenge-surface-detection/train_labels')
    if train_label_dir.exists():
        files = sorted(train_label_dir.glob('*.tif'))
        if files:
            arr = tifffile.imread(str(files[0]))
            return arr.dtype
    return np.uint8


def run_submission(models, test_ids, cfg) -> Path:
    """Public interface requested. Writes /kaggle/working/submission.zip."""
    dtype = _infer_mask_dtype()
    runtime_overlap = cfg.OVERLAP
    runtime_tta = cfg.TTA_MODE

    times = []
    n_total = len(test_ids)

    with zipfile.ZipFile(cfg.SUBMISSION_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for idx, image_id in enumerate(test_ids, start=1):
            tif_path = cfg.TEST_IMAGES_DIR / f'{image_id}.tif'
            t0 = time.time()
            volume = tifffile.imread(str(tif_path))

            prob_map = weighted_ensemble_inference(
                models=models,
                weights=fold_weights,
                volume=volume,
                cfg=cfg,
                overlap=runtime_overlap,
                tta_mode=runtime_tta,
            )

            pred = apply_postprocess(
                prob_map,
                threshold=cfg.PP_THRESHOLD,
                use_hysteresis=cfg.PP_USE_HYSTERESIS,
                hysteresis_low=cfg.PP_HYST_LOW,
                hysteresis_high=cfg.PP_HYST_HIGH,
                min_component_size=cfg.PP_MIN_COMPONENT_SIZE,
                closing_iters=cfg.PP_CLOSING_ITERS,
            )

            out_path = cfg.OUTPUT_DIR / f'{image_id}.tif'
            tifffile.imwrite(out_path, pred.astype(dtype))
            zf.write(out_path, arcname=f'{image_id}.tif')
            out_path.unlink()

            dt = time.time() - t0
            times.append(dt)
            print(f'[{idx}/{n_total}] {image_id} | {dt:.1f}s | overlap={runtime_overlap:.2f} | tta={runtime_tta}')

            if idx == min(cfg.RUNTIME_GATE_CASES, n_total):
                projected_hours = (np.mean(times) * n_total) / 3600.0
                print(f'Runtime projection: {projected_hours:.2f}h (limit {cfg.RUNTIME_HOURS_LIMIT}h)')
                if projected_hours > cfg.RUNTIME_HOURS_LIMIT:
                    if runtime_tta != 'none':
                        print('Runtime gate: disabling TTA first')
                        runtime_tta = 'none'
                    elif runtime_overlap > cfg.MIN_OVERLAP:
                        new_overlap = max(cfg.MIN_OVERLAP, runtime_overlap - 0.1)
                        print(f'Runtime gate: reducing overlap {runtime_overlap:.2f} -> {new_overlap:.2f}')
                        runtime_overlap = new_overlap

            del volume, prob_map, pred
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    return cfg.SUBMISSION_ZIP


print()
print('=' * 80)
print('SUBMISSION GENERATION')
print('=' * 80)
submission_path = run_submission(models=models, test_ids=test_df['id'].astype(int).tolist(), cfg=cfg)
print('Submission zip:', submission_path)
print('=' * 80)


In [ ]:
# Verify
with zipfile.ZipFile(cfg.SUBMISSION_ZIP, 'r') as zf:
    print(f"Files: {len(zf.namelist())}")
    for f in zf.namelist():
        print(f"  {f}: {zf.getinfo(f).file_size/1e6:.2f} MB")

print("\nReady to submit!")